# 1. Environment Setup and Library Installation
First, we need to install the necessary libraries from Hugging Face and others required for translation metrics (SacreBLEU) and experiment tracking (WandB).


In [ ]:
!pip install -q transformers datasets sacrebleu accelerate evaluate -U

# 2. Imports and Reproducibility
Here we import the standard libraries. Crucially, we set random seeds for torch, numpy, and random. This ensures that our experiments are reproducible—if you run this code twice, the model initialization and data shuffling will remain the same.

We also handle logins for Weights & Biases (WandB) for tracking loss curves and Hugging Face for pushing the final model.

In [ ]:
# ===============================
# 1) Imports
# ===============================
import os
import torch
import datasets
import sacrebleu
import pandas as pd
import numpy as np
import random
from datasets import Dataset, DatasetDict
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)
import evaluate

# Set seeds for consistency across experiments
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Setting up wandb and hugging face
import wandb
os.environ["WANDB_NOTEBOOK_NAME"] = "m2m100-7e-5-20k-Ultimate.ipynb"

# !!! REPLACE WITH YOUR KEYS !!!
os.environ["WANDB_API_KEY"] = "your_wandb_api_key_here"
wandb.login()

from huggingface_hub import login
login(token="your_hugging_face_token_here")

# 3. Data Loading
We load the parallel text files (TSV or line-by-line text).

Source: Nepali (ne)

Target: Tamang (tmg)

The function below reads the files, ensures line counts match, and wraps them into a Hugging Face DatasetDict.

In [ ]:
# ===============================
# 2) Load parallel TSVs
# ===============================
def load_parallel(src_file, tgt_file, src_key="ne", tgt_key="tmg"):
    with open(src_file, "r", encoding="utf-8") as f:
        src_lines = [l.rstrip("\n") for l in f.readlines()]
    with open(tgt_file, "r", encoding="utf-8") as f:
        tgt_lines = [l.rstrip("\n") for l in f.readlines()]
    assert len(src_lines) == len(tgt_lines), f"Counts differ: {src_file} vs {tgt_file}"
    return {src_key: src_lines, tgt_key: tgt_lines}

# Ensure these paths are correct for your environment
train_data = load_parallel("/content/ne-train.tsv", "/content/tmg-train.tsv", "ne", "tmg")
test_data = load_parallel("/content/ne-test.tsv", "/content/tmg-test.tsv", "ne", "tmg")

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "test": Dataset.from_dict(test_data)
})
print("Loaded dataset splits:", {k: len(v) for k, v in dataset.items()})

# 4. Model Initialization & Language Proxy Strategy
We load the M2M100 418M parameter model.

Critical Detail: M2M100 does not natively support Tamang (tmg). To bypass this, we use Hindi (hi) as a proxy language token. Since Tamang and Hindi share the Devanagari script and some linguistic features, using the pre-trained Hindi embeddings provides a better starting point than random initialization.

In [ ]:
# ===============================
# 3) Load model + tokenizer
# ===============================
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Required settings to save VRAM and enable gradient checkpointing
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Tamang uses Hindi embedding (Language Proxy Strategy)
tmg_lang_id_forcing = tokenizer.get_lang_id("hi")
ne_lang_id_forcing = tokenizer.get_lang_id("ne")

tokenizer.model_max_length = 128

# 5. Bidirectional PreprocessingTo create a robust translation model, we train
NE -> TMG: We map Nepali inputs to Tamang labels.
TMG -> NE: We map Tamang inputs to Nepali labels.We concatenate these two datasets effectively doubling our training data and allowing the model to translate in both directions.

In [ ]:
# ===============================
# 4) Preprocess
# ===============================
max_length = 128

def preprocess_fn(ex, src_key, tgt_key):
    inputs = tokenizer(
        ex[src_key],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    targets = tokenizer(
        ex[tgt_key],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

# Create Forward Direction (NE -> TMG)
train_ne2tmg = dataset["train"].map(lambda x: preprocess_fn(x, "ne", "tmg"), remove_columns=["ne", "tmg"])
test_ne2tmg = dataset["test"].map(lambda x: preprocess_fn(x, "ne", "tmg"), remove_columns=["ne", "tmg"])

# Create Backward Direction (TMG -> NE)
train_tmg2ne = dataset["train"].map(lambda x: preprocess_fn(x, "tmg", "ne"), remove_columns=["ne", "tmg"])
test_tmg2ne = dataset["test"].map(lambda x: preprocess_fn(x, "tmg", "ne"), remove_columns=["ne", "tmg"])

# Concatenate for single training loop
train_dataset = datasets.concatenate_datasets([train_ne2tmg, train_tmg2ne])
test_dataset = datasets.concatenate_datasets([test_ne2tmg, test_tmg2ne])

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

# 6. Training Configuration
We configure the Trainer. Key parameters:

Learning Rate: 7e-5 (Typical for M2M100 fine-tuning).

FP16: Enabled for faster training on GPUs.

Push to Hub: Automatically uploads the model to Hugging Face at the end of training.

Metric: We track Loss to decide the "best" model.

In [ ]:
# ===============================
# 5) Data collator
# ===============================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# ===============================
# 6) Training args
# ===============================
repo_name = "repo_name_here"

training_args = TrainingArguments(
    output_dir="./m2m100_trained",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=7e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,                # Mixed precision for speed
    save_total_limit=1,       # Only keep the last checkpoint to save space
    remove_unused_columns=False,
    push_to_hub=True,
    hub_model_id=repo_name,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="wandb",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# 7. Execution and Deployment
Finally, we start the training loop. Once finished, we save the model locally and push the final weights and tokenizer to the Hugging Face Hub for easy inference later.

In [ ]:
# ===============================
# 8) Train
# ===============================
trainer.train()

# Save locally
trainer.save_model("./m2m100_trained")
tokenizer.save_pretrained("./m2m100_trained")

# Push to Hugging Face Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model and tokenizer pushed to Hugging Face: {repo_name}")
print("Model saved to ./m2m100_trained")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompa

wandb: WARNING WANDB_NOTEBOOK_NAME should be a path to a notebook file, couldn't find m2m100-7e-5-20k-Ultimate.ipynb.
wandb: Currently logged in as: rupaktiwari (rupaktiwari-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded dataset splits: {'train': 15001, 'test': 5000}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Step,Training Loss
3751,0.844800
7502,0.210500
11253,0.143900
15004,0.096100
18755,0.062400


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model and tokenizer pushed to Hugging Face: wandererupak/m2m100-7e-5-20k-Ultimate
✅ Model saved to ./m2m100_trained


# M2M100 Bidirectional Evaluation Summary
This code block performs the final, comprehensive evaluation of the fine-tuned M2M100 model for Nepali-Tamang (NE-TMG) translation, covering both translation directions.
1. Setup & Model LoadingInstalls unbabel-comet and loads the fine-tuned model and tokenizer from ./m2m100_trained.The model is moved to the GPU/CPU and set to eval() mode.
2. Inference StrategyThe generate_batch function handles fast, batched translation.Crucial Language Proxy: Since M2M100 lacks a native Tamang code, the script uses the Hindi language ID (hi) as a token proxy for Tamang sentences (source or target).
3. Metrics CalculatedThe script loads and computes five standard machine translation metrics:BLEU: N-gram overlap.chrF/chrF++: Character-level F-scores (robust for morphologically rich languages).METEOR: Based on alignment, stemming, and synonymy.COMET: A neural metric highly correlated with human judgment.
4. Bidirectional ResultsThe evaluation is executed and results are printed for two directions:Nepali to Tamang: (Source ne, Target forced to hi proxy).Tamang to Nepali: (Source hi proxy, Target forced to ne).The final output provides a quantitative score for each metric in both directions, followed by a small qualitative sample of source, prediction, and reference translations for visual inspection.

In [ ]:
!pip install unbabel-comet
import sacrebleu
import evaluate
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

# Load the trained model and tokenizer
model_path = "./m2m100_trained"  # Use final saved model
model = M2M100ForConditionalGeneration.from_pretrained(model_path)
tokenizer = M2M100Tokenizer.from_pretrained(model_path)
print(f"✅ Loaded model and tokenizer from {model_path}")

# Move model to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode
print(f"Model is on device: {device}")

# ===============================
# Fast batched inference + BLEU, chrF, chrF++, METEOR, COMET (both directions)
# ===============================
def generate_batch(sentences, src_lang, forced_bos_token_id, batch_size=32, max_length=128):
    all_preds = []
    tokenizer.src_lang = src_lang
    device = next(model.parameters()).device
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        # Enable use_cache during generation for faster decoding
        prev = model.config.use_cache
        model.config.use_cache = True
        gen = model.generate(**enc, forced_bos_token_id=forced_bos_token_id, max_length=max_length)
        model.config.use_cache = prev
        dec = tokenizer.batch_decode(gen, skip_special_tokens=True)
        all_preds.extend([d.strip() for d in dec])
    return all_preds

def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f.readlines()]

# Load test data
ne_test_lines = load_lines("/content/ne-test.tsv")
tmg_test_lines = load_lines("/content/tmg-test.tsv")
assert len(ne_test_lines) == len(tmg_test_lines), "Test files must be aligned"

# Debug: Print test set size and sample data
print(f"Test set size: Nepali={len(ne_test_lines)}, Tamang={len(tmg_test_lines)}")
print("Sample Test Data (First 3):")
for i in range(min(3, len(ne_test_lines))):
    print(f"NE: {ne_test_lines[i]}")
    print(f"TMG: {tmg_test_lines[i]}")
    print("---")

# Initialize metrics
meteor = evaluate.load("meteor")
comet = evaluate.load("comet", "Unbabel/wmt22-comet-da")

# Nepali -> Tamang
tmg_lang_id_forcing = tokenizer.get_lang_id("hi")  # Hindi hack for Tamang
preds_ne2tmg = generate_batch(ne_test_lines, src_lang="ne", forced_bos_token_id=tmg_lang_id_forcing, batch_size=16)

# Debug: Print sample predictions
print("Sample Nepali -> Tamang Predictions (First 3):")
for i in range(min(3, len(preds_ne2tmg))):
    print(f"Input NE: {ne_test_lines[i]}")
    print(f"Pred TMG: {preds_ne2tmg[i]}")
    print(f"Ref TMG: {tmg_test_lines[i]}")
    print("---")

# Calculate metrics for Nepali -> Tamang
bleu_ne2tmg = sacrebleu.corpus_bleu(preds_ne2tmg, [tmg_test_lines]).score
chrf_ne2tmg = sacrebleu.corpus_chrf(preds_ne2tmg, [tmg_test_lines]).score
chrfpp_ne2tmg = sacrebleu.corpus_chrf(preds_ne2tmg, [tmg_test_lines], word_order=2).score
meteor_results_ne2tmg = meteor.compute(predictions=preds_ne2tmg, references=tmg_test_lines)
meteor_ne2tmg = meteor_results_ne2tmg["meteor"] * 100  # Convert to 0-100 scale
comet_results_ne2tmg = comet.compute(predictions=preds_ne2tmg, references=tmg_test_lines, sources=ne_test_lines)
comet_ne2tmg = comet_results_ne2tmg["mean_score"] * 100  # Convert to 0-100 scale

print(f"Nepali → Tamang Metrics:")
print(f"  BLEU: {bleu_ne2tmg:.2f}")
print(f"  chrF: {chrf_ne2tmg:.2f}")
print(f"  chrF++: {chrfpp_ne2tmg:.2f}")
print(f"  METEOR: {meteor_ne2tmg:.2f}")
print(f"  COMET: {comet_ne2tmg:.2f}")

# Tamang -> Nepali
ne_lang_id_forcing = tokenizer.get_lang_id("ne")
preds_tmg2ne = generate_batch(tmg_test_lines, src_lang="hi", forced_bos_token_id=ne_lang_id_forcing, batch_size=16)

# Debug: Print sample predictions
print("Sample Tamang -> Nepali Predictions (First 3):")
for i in range(min(3, len(preds_tmg2ne))):
    print(f"Input TMG: {tmg_test_lines[i]}")
    print(f"Pred NE: {preds_tmg2ne[i]}")
    print(f"Ref NE: {ne_test_lines[i]}")
    print("---")

# Calculate metrics for Tamang -> Nepali
bleu_tmg2ne = sacrebleu.corpus_bleu(preds_tmg2ne, [ne_test_lines]).score
chrf_tmg2ne = sacrebleu.corpus_chrf(preds_tmg2ne, [ne_test_lines]).score
chrfpp_tmg2ne = sacrebleu.corpus_chrf(preds_tmg2ne, [ne_test_lines], word_order=2).score
meteor_results_tmg2ne = meteor.compute(predictions=preds_tmg2ne, references=ne_test_lines)
meteor_tmg2ne = meteor_results_tmg2ne["meteor"] * 100  # Convert to 0-100 scale
comet_results_tmg2ne = comet.compute(predictions=preds_tmg2ne, references=ne_test_lines, sources=tmg_test_lines)
comet_tmg2ne = comet_results_tmg2ne["mean_score"] * 100  # Convert to 0-100 scale

print(f"Tamang → Nepali Metrics:")
print(f"  BLEU: {bleu_tmg2ne:.2f}")
print(f"  chrF: {chrf_tmg2ne:.2f}")
print(f"  chrF++: {chrfpp_tmg2ne:.2f}")
print(f"  METEOR: {meteor_tmg2ne:.2f}")
print(f"  COMET: {comet_tmg2ne:.2f}")

# Print sample predictions
print("\nSample Predictions:")
for i in range(5):
    print(f"Example {i+1}:")
    print("NE:", ne_test_lines[i])
    print("PRED -> TMG:", preds_ne2tmg[i])
    print("REF  -> TMG:", tmg_test_lines[i])
    print("---")
    print("TMG:", tmg_test_lines[i])
    print("PRED -> NE:", preds_tmg2ne[i])
    print("REF  -> NE:", ne_test_lines[i])
    print("=====")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 21.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not current

✅ Loaded model and tokenizer from ./m2m100_trained
Model is on device: cuda
Test set size: Nepali=5000, Tamang=5000
Sample Test Data (First 3):
NE: होटल बेचेर यहाँ आइयो।
TMG: होटल चुङ्सि चुरि खाजि।
---
NE: सुनिदिने निकाय मौन छ।
TMG: थाइतोबा निकाय कुटिसि चिबा मुला।
---
NE: अहिले फेरि मेन्था रोप्दैछौँ।
TMG: दान्दे दोसि मेन्था सुबान मुला।
---


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

...

Sample Nepali -> Tamang Predictions (First 3):
Input NE: होटल बेचेर यहाँ आइयो।
Pred TMG: होटल चुङ्सि चुरि खाजि।
Ref TMG: होटल चुङ्सि चुरि खाजि।
---
Input NE: सुनिदिने निकाय मौन छ।
Pred TMG: थेसि पिन्बा निकाय कुटिसि मुला।
Ref TMG: थाइतोबा निकाय कुटिसि चिबा मुला।
---
Input NE: अहिले फेरि मेन्था रोप्दैछौँ।
Pred TMG: दान्दे दोःसि मेन्था सुबाकेन मुला।
Ref TMG: दान्दे दोसि मेन्था सुबान मुला।
---


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Nepali → Tamang Metrics:
  BLEU: 40.24
  chrF: 73.30
  chrF++: 69.27
  METEOR: 60.81
  COMET: 75.67
Sample Tamang -> Nepali Predictions (First 3):
Input TMG: होटल चुङ्सि चुरि खाजि।
Pred NE: होटल समातेर यहाँ आयो।
Ref NE: होटल बेचेर यहाँ आइयो।
---
Input TMG: थाइतोबा निकाय कुटिसि चिबा मुला।
Pred NE: सुनिनुपर्ने निकाय चुपचाप बसेको छ।
Ref NE: सुनिदिने निकाय मौन छ।
---
Input TMG: दान्दे दोसि मेन्था सुबान मुला।
Pred NE: अहिले फेरि मेन्था लगाउँदै छ।
Ref NE: अहिले फेरि मेन्था रोप्दैछौँ।
---


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Tamang → Nepali Metrics:
  BLEU: 42.73
  chrF: 72.21
  chrF++: 68.74
  METEOR: 61.07
  COMET: 79.13

Sample Predictions:
Example 1:
NE: होटल बेचेर यहाँ आइयो।
PRED -> TMG: होटल चुङ्सि चुरि खाजि।
REF  -> TMG: होटल चुङ्सि चुरि खाजि।
---
TMG: होटल चुङ्सि चुरि खाजि।
PRED -> NE: होटल समातेर यहाँ आयो।
REF  -> NE: होटल बेचेर यहाँ आइयो।
=====
Example 2:
NE: सुनिदिने निकाय मौन छ।
PRED -> TMG: थेसि पिन्बा निकाय कुटिसि मुला।
REF  -> TMG: थाइतोबा निकाय कुटिसि चिबा मुला।
---
TMG: थाइतोबा निकाय कुटिसि चिबा मुला।
PRED -> NE: सुनिनुपर्ने निकाय चुपचाप बसेको छ।
REF  -> NE: सुनिदिने निकाय मौन छ।
=====
Example 3:
NE: अहिले फेरि मेन्था रोप्दैछौँ।
PRED -> TMG: दान्दे दोःसि मेन्था सुबाकेन मुला।
REF  -> TMG: दान्दे दोसि मेन्था सुबान मुला।
---
TMG: दान्दे दोसि मेन्था सुबान मुला।
PRED -> NE: अहिले फेरि मेन्था लगाउँदै छ।
REF  -> NE: अहिले फेरि मेन्था रोप्दैछौँ।
=====
Example 4:
NE: यो नकारात्मक पक्ष हो।
PRED -> TMG: चु नकारात्मक पक्ष हिन्ना।
REF  -> TMG: चु नकरात्मक तिलो हिन्ना।
---
TMG: चु नकरात्मक तिलो हिन्ना।
